# Text Cleaning Pipeline

Cleans `siraplimau.com.jsonl` (Malay-language gaming news, `url` / `title` / `body`) for downstream
embedding / language-model use.

Pipeline stages:

1. Structural / boilerplate cleanup (site-specific: TikTok footer, tag lists, tweet/Facebook embed junk, image-carousel widgets, source stubs)
2. Language / content filtering (keep Malay/Indonesian text, drop junk/near-empty/foreign-script pages)
3. Normalization (Unicode NFC, whitespace collapsing, punctuation handling, cased + lowercased variants)
4. Sentence & word tokenization (Malaya if available, regex fallback otherwise)
5. Stopword removal (built as an **optional side artifact** — off by default since the target is embeddings/LLM input, not BoW/TF-IDF)
6. Lemmatization via [Malaya](https://github.com/huseinzol05/malaya) (`malaya.stem`)
7. Manual tagging scaffold (auto-suggested categories + exportable annotation template)

Each stage is its own section so you can inspect intermediate output before moving on.

## 0. Setup & configuration

In [1]:
# Core dependencies for this notebook. Installs into whichever kernel is currently
# selected (safe to re-run — pip skips anything already satisfied).
# %pip install --quiet pandas tqdm fast-langdetect pyarrow

# Malaya, for real Malay-aware tokenization/stopwords/lemmatization (regex fallbacks are
# used automatically if this is skipped). Uncomment to enable — it's a heavier install:
# torch (CPU build) + PySastrawi are required alongside malaya itself.
# %pip install --quiet malaya PySastrawi
# %pip install --quiet torch --index-url https://download.pytorch.org/whl/cpu
# %pip install --quiet ipywidgets

import json
import re
import unicodedata
import functools
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

# ---- Config -----------------------------------------------------------
DATA_PATH = Path("../data/raw/siraplimau.com.jsonl")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The full corpus is ~13k docs. Heavy steps (tokenization, lemmatization)
# support running on a sample first — set to None to run on everything.
SAMPLE_SIZE = None          # e.g. 500 for a quick pass, None for full corpus
RANDOM_STATE = 42

MIN_BODY_CHARS = 80         # drop docs with less real content than this after structural cleanup
DO_STOPWORD_REMOVAL = False  # embeddings/LM input -> keep stopwords by default


c:\Users\User\Desktop\NLP\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Install accelerate for GPU device mapping (required for transformers device_map="auto")
%pip install --quiet accelerate

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Verify GPU setup and installation
import torch

print("=" * 60)
print("PYTORCH & GPU VERIFICATION")
print("=" * 60)
print(f"\n✓ PyTorch Version: {torch.__version__}")
print(f"✓ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ CUDA Version: {torch.version.cuda}")
    print(f"✓ cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"✓ Number of GPUs: {torch.cuda.device_count()}")
    print(f"\nGPU Details:")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}:")
        print(f"    Name: {props.name}")
        print(f"    Capability: {props.major}.{props.minor}")
        print(f"    Total Memory: {props.total_memory / 1e9:.2f} GB")
    print(f"\n✓ Current GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Index: {torch.cuda.current_device()}")
    
    # Test GPU memory
    try:
        test_tensor = torch.randn(1000, 1000).cuda()
        print(f"✓ GPU Memory Test: PASSED (tensor created on GPU)")
        print(f"✓ Allocated GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"✓ Cached GPU Memory: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
        del test_tensor
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"✗ GPU Memory Test: FAILED - {e}")
else:
    print("\n⚠ WARNING: CUDA/GPU not detected! Training will run on CPU (much slower).")
    print("\nTo use GPU, ensure:")
    print("  1. NVIDIA GPU is installed on your system")
    print("  2. NVIDIA drivers are installed")
    print("  3. CUDA toolkit is installed")

print("\n" + "=" * 60)
print("Ready for GPU training!" if torch.cuda.is_available() else "Running on CPU mode")
print("=" * 60)

PYTORCH & GPU VERIFICATION

✓ PyTorch Version: 2.7.1+cu128
✓ CUDA Available: True
✓ CUDA Version: 12.8
✓ cuDNN Version: 90701
✓ Number of GPUs: 1

GPU Details:
  GPU 0:
    Name: NVIDIA GeForce RTX 5090
    Capability: 12.0
    Total Memory: 34.19 GB

✓ Current GPU: NVIDIA GeForce RTX 5090
✓ GPU Index: 0
✓ GPU Memory Test: PASSED (tensor created on GPU)
✓ Allocated GPU Memory: 0.00 GB
✓ Cached GPU Memory: 0.02 GB

Ready for GPU training!


## 1. Load data

Quick look at the raw corpus before touching anything.

In [4]:
records = []
with DATA_PATH.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"rows: {len(df)}")
print(f"columns: {list(df.columns)}")
print(f"missing body: {df['body'].isna().sum()}")
print(f"duplicate urls: {df.duplicated('url').sum()}")
df.head(3)


rows: 8504
columns: ['url', 'title', 'body']
missing body: 0
duplicate urls: 0


,url,title,body
0,https://siraplimau.com/1-inci-pun-boleh-lemas-...,‘1 Inci Pun Boleh Lemas’- Ini Cara Elak Risiko...,Insiden kanak-kanak lemas di dalam bilik air b...
1,https://siraplimau.com/1-kebaikan-dapat-upah-r...,‘1 Kebaikan Dapat Upah RM1’ – Lelaki Kongsi Te...,"“Papa,papa..jom pergi kedai nak beli mainan…jo..."
2,https://siraplimau.com/1-minuman-2-rasa-susu-f...,1 Minuman 2 Rasa – Susu Frog Bang Dari Tealive...,Ha! Kalau nak tahu baru-baru ni admin dijemput...


In [5]:
# Fill missing bodies with empty string so string ops don't choke later
df["body"] = df["body"].fillna("")
df["title"] = df["title"].fillna("")
df["body_len_raw"] = df["body"].str.len()
df["body_len_raw"].describe()


count     8504.000000
mean      3266.591722
std       1881.934494
min         49.000000
25%       1945.750000
50%       2814.000000
75%       4118.500000
max      30722.000000
Name: body_len_raw, dtype: float64

## 2. Structural / boilerplate cleanup (site-specific)

Inspecting random samples of articles shows a handful of recurring,
non-content patterns baked into every page by the CMS/embed widgets:

- **Footer boilerplate** — every article ends with `Jangan lupa untuk ikuti kami di TikTok!`,
  the `@gamersantaimy` handle, and a `Tags: ...` line with concatenated tag keywords
  (no separators between tags). We strip this and pull the tags out into their own column.
- **Embedded tweet debris** — `pic.twitter.com/xxxx` links followed by an
  `— Author (@handle) Month Day, Year` attribution line, left over from embedded tweet widgets
  whose actual HTML/JS didn't survive scraping.
- **Embedded Facebook/Instagram post debris** — similar leftover caption/attribution lines
  from social embeds (e.g. `... さんの投稿`, `Posted by ...`).
- **Image-carousel widget junk** — scraped image-gallery sliders leave behind lines that are
  just pagination controls: `1 of  8`, lone `-`/`+` characters, and long runs of blank lines
  where `<img>` tags used to be.
- **Bare source stubs** — `Sumber: XYZ` / `Sumber : XYZ` lines are attribution stubs, not
  sentences; they break sentence tokenization if left in, so they're removed (the source
  itself isn't part of the article's semantic content).
- **Non-breaking spaces (`\xa0`)** left by the CMS.


In [6]:
FOOTER_RE = re.compile(
    r"Jangan lupa untuk ikuti kami di TikTok!.*?"
    r"(?:Tags:\s*(?P<tags>.*?))?\s*$",
    re.DOTALL,
)

TWEET_JUNK_RE = re.compile(
    r"^.*pic\.twitter\.com/\S+\s*\n?"      # pic.twitter.com/xxxx line
    r"(?:^—.*$\n?)?",                          # optional "— Author (@handle) Date" line after it
    re.MULTILINE,
)

SOCIAL_EMBED_RE = re.compile(
    r"^.*(さんの投稿|Posted by|shared a post).*$\n?",
    re.MULTILINE,
)

CAROUSEL_JUNK_RE = re.compile(
    r"^\s*(\d+\s+of\s+\d+|[-+])\s*$\n?",
    re.MULTILINE,
)

SOURCE_STUB_RE = re.compile(
    r"^\s*Sumber\s*:\s*.*$\n?",
    re.MULTILINE,
)

MULTI_BLANK_RE = re.compile(r"\n\s*\n\s*\n+")


def structural_clean(body: str) -> tuple[str, str]:
    """Returns (cleaned_body, extracted_tags_string)."""
    text = body.replace("\xa0", " ")

    tags = ""
    m = FOOTER_RE.search(text)
    if m:
        tags = (m.group("tags") or "").strip()
        text = text[: m.start()]

    text = TWEET_JUNK_RE.sub("\n", text)
    text = SOCIAL_EMBED_RE.sub("\n", text)
    text = CAROUSEL_JUNK_RE.sub("\n", text)
    text = SOURCE_STUB_RE.sub("\n", text)

    # collapse 3+ blank lines (left behind by stripped images/widgets) down to one
    text = MULTI_BLANK_RE.sub("\n\n", text)
    text = text.strip()
    return text, tags


cleaned = df["body"].progress_apply(structural_clean)
df["body_struct"] = cleaned.apply(lambda t: t[0])
df["tags_site"] = cleaned.apply(lambda t: t[1])
df["body_len_struct"] = df["body_struct"].str.len()

print("median raw length   :", df["body_len_raw"].median())
print("median struct length:", df["body_len_struct"].median())
print("chars removed (median):", (df["body_len_raw"] - df["body_len_struct"]).median())
df[["title", "body_len_raw", "body_len_struct", "tags_site"]].sample(5, random_state=RANDOM_STATE)


100%|██████████| 8504/8504 [00:00<00:00, 11702.52it/s]

median raw length   : 2814.0
median struct length: 2775.5
chars removed (median): 39.0


,title,body_len_raw,body_len_struct,tags_site
5164,9 Sebab Mengapa Anda Kerap Sembelit dan Cara M...,3848,3830,
7362,Rebut Tawaran Sehingga RM1.5 juta Bersama LBS ...,3535,3512,
3469,"Elak Takutkan Anak-Anak Dengan Ketinggian, Gel...",2218,2126,
3979,Puasa Israk Mikraj (27 Rejab) Bukan Termasuk P...,7378,7352,
5292,"Mudah Tanam, Buah & Daun Semua Boleh Makan. In...",4940,4922,


In [7]:
# Drop docs that are empty / near-empty after boilerplate removal, and exact duplicates
before = len(df)
df = df[df["body_struct"].str.len() >= MIN_BODY_CHARS].copy()
df = df.drop_duplicates(subset="body_struct")
print(f"dropped {before - len(df)} rows (too short after cleanup, or exact duplicate) -> {len(df)} remain")


dropped 4 rows (too short after cleanup, or exact duplicate) -> 8500 remain


## 3. Language / content filtering

Even after structural cleanup, a few pages remain that are effectively non-content
(stray widget captions, code dumps, or articles that are mostly English/Japanese with
no real Malay text). We use `fast_langdetect` (fastText under the hood) to keep only
docs whose dominant language is Malay/Indonesian, and add a cheap non-Latin-script
ratio check to catch leftover CJK widget junk that language ID alone might miss.


In [8]:
_detect_failures = []  # collect (text, exception) so failures are visible, not silently swallowed

try:
    from fast_langdetect import detect as _ft_detect

    def detect_lang(text: str) -> str:
        sample = text[:1000] if text else ""
        if not sample.strip():
            return "unknown"
        try:
            result = _ft_detect(sample.replace("\n", " "))
            # fast_langdetect's return shape changed across versions:
            # 0.3.x -> a single dict {"lang": ..., "score": ...}
            # 1.x   -> a list of dicts [{"lang": ..., "score": ...}, ...]
            if isinstance(result, list):
                result = result[0]
            return result["lang"]
        except Exception as e:
            _detect_failures.append((sample[:80], repr(e)))
            return "unknown"

except ImportError:
    # Fallback: crude function-word overlap heuristic if fast_langdetect isn't installed
    _MALAY_MARKERS = {"yang", "dan", "di", "untuk", "dengan", "ini", "itu", "dalam", "tidak", "akan"}

    def detect_lang(text: str) -> str:
        words = re.findall(r"[a-zA-Z]+", text.lower()[:1000])
        if not words:
            return "unknown"
        overlap = sum(1 for w in words if w in _MALAY_MARKERS) / len(words)
        return "ms" if overlap > 0.03 else "other"


NON_LATIN_RE = re.compile(r"[぀-ヿ㐀-鿿가-힯]")  # CJK/Hangul/Kana


def non_latin_ratio(text: str) -> float:
    if not text:
        return 0.0
    return len(NON_LATIN_RE.findall(text)) / max(len(text), 1)


df["lang"] = df["body_struct"].progress_apply(detect_lang)
df["non_latin_ratio"] = df["body_struct"].apply(non_latin_ratio)

print("Language distribution:")
print(df["lang"].value_counts())

if _detect_failures:
    print(f"\n{len(_detect_failures)} detection calls raised an exception (treated as 'unknown'). First few:")
    for text, err in _detect_failures[:3]:
        print(f"  {err}  <- {text!r}")


100%|██████████| 8500/8500 [00:00<00:00, 25365.80it/s]


Language distribution:
lang
id    5291
ms    3091
en      99
tr       5
pl       3
it       2
de       2
tl       2
jv       1
uz       1
hu       1
eo       1
su       1
Name: count, dtype: int64


In [9]:
KEEP_LANGS = {"ms", "id", "unknown"}  # Malay / Indonesian, plus "unknown" (detection failed -> fail safe, don't drop)

before = len(df)
dropped_preview = df[~(df["lang"].isin(KEEP_LANGS) & (df["non_latin_ratio"] < 0.05))]
df = df[df["lang"].isin(KEEP_LANGS) & (df["non_latin_ratio"] < 0.05)].copy()
print(f"dropped {before - len(df)} rows (confidently non-Malay language, or too much non-Latin script) -> {len(df)} remain")

if len(dropped_preview):
    print("\nSample of dropped rows (sanity-check these are actually not Malay content):")
    print(dropped_preview[["title", "lang", "non_latin_ratio"]].head(10).to_string())


dropped 120 rows (confidently non-Malay language, or too much non-Latin script) -> 8380 remain

Sample of dropped rows (sanity-check these are actually not Malay content):
                                                                                                                                         title lang  non_latin_ratio
102                                                                   11street Rai Ulang Tahun Pertama Dengan Tawaran ‘Shocking Deals To Life’   en              0.0
141                                                                                   2 Anggota Polis Jadi Wira Sambut Bayi Lahir Dalam Kereta   en              0.0
410                                                              6 Barang Yang Tak Disangka Berguna Pada Hari Raya. Yang Last Tu Memang Betul!   en              0.0
449                                                                                     Jaga-jaga, Ini 6 Tanda Anda Mula Bosan Dengan Pasangan   en              0.0
493

In [10]:
# KEEP_LANGS = {"ms", "id", "unknown"}  # Malay / Indonesian, plus "unknown" (detection failed -> fail safe, don't drop)

# before = len(df)
# dropped_df = df[~(df["lang"].isin(KEEP_LANGS) & (df["non_latin_ratio"] < 0.05))].copy()
# df = df[df["lang"].isin(KEEP_LANGS) & (df["non_latin_ratio"] < 0.05)].copy()
# print(f"dropped {before - len(df)} rows (confidently non-Malay language, or too much non-Latin script) -> {len(df)} remain")

# # Save dropped rows to files in OUT_DIR
# if len(dropped_df):
#     dropped_csv_path = OUT_DIR / "dropped_non_malay.csv"
#     dropped_jsonl_path = OUT_DIR / "dropped_non_malay.jsonl"
#     dropped_df.to_csv(dropped_csv_path, index=False, encoding="utf-8")
#     print(f"\nSaved {len(dropped_df)} dropped rows to:")
#     print(f"  - {dropped_csv_path}")

#     print("\nSample of dropped rows (sanity-check these are actually not Malay content):")
#     print(dropped_df[["title", "lang", "non_latin_ratio"]].head(10).to_string())


## 4. Normalization

- **Unicode NFC** normalization (composes accented chars, canonicalizes look-alike code points).
- **Whitespace collapsing** — all runs of whitespace -> single space, trimmed.
- **Punctuation handling** — curly quotes/dashes/ellipses normalized to plain ASCII equivalents,
  runs of repeated punctuation (`!!!`, `??`) collapsed.
- **Case** — we keep a `*_cased` (original-case, for NER / proper-noun detection later) and a
  `*_lower` variant (for lemmatization input) of both `title` and `body`.


In [11]:
PUNCT_MAP = {
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-", "\u2026": "...",
}
PUNCT_TRANS = str.maketrans(PUNCT_MAP)

REPEAT_PUNCT_RE = re.compile(r"([!?.,])\1{1,}")
WS_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.translate(PUNCT_TRANS)
    text = REPEAT_PUNCT_RE.sub(r"\1", text)
    text = WS_RE.sub(" ", text).strip()
    return text


df["body_cased"] = df["body_struct"].progress_apply(normalize_text)
df["body_lower"] = df["body_cased"].str.lower()
df["title_cased"] = df["title"].apply(normalize_text)
df["title_lower"] = df["title_cased"].str.lower()

df[["title_cased", "body_cased"]].sample(2, random_state=RANDOM_STATE)


100%|██████████| 8380/8380 [00:01<00:00, 7712.52it/s]


,title_cased,body_cased
2891,"Cara Tutup Setting Di Facebook, Baru Kurang Ri...","Kebelakangan ini, terlalu banyak isu penggodam..."
8295,"Lafaz Taklik Dibelakang Sijil Nikah, Tersembun...",Isu isteri menjadi mangsa penderaan semakin me...


## 5. Sentence & word tokenization

Uses Malaya's tokenizer when available (`malaya.tokenizer.Tokenizer` for words,
`malaya.text.function.split_into_sentences` for sentences — both tuned for Malay
abbreviations/particles). Falls back to a regex-based tokenizer otherwise so the
notebook still runs end-to-end without the dependency installed.

Word tokens are built from `body_lower` (lemmatization input); sentences are kept from
`body_cased` since sentence boundaries/proper nouns read better in original case.

This step is run on a sample (`SAMPLE_SIZE`) by default — it's the first properly
expensive stage. Set `SAMPLE_SIZE = None` above to run the full corpus.


In [12]:
import warnings

# Malaya's tokenizer regex uses a deprecated nested-set pattern internally (upstream issue,
# not our code) -> harmless, silence it so it doesn't clutter the output.
warnings.filterwarnings("ignore", category=FutureWarning, module="malaya")

try:
    import malaya

    _word_tokenizer = malaya.tokenizer.Tokenizer()

    def sentence_tokenize(text: str) -> list[str]:
        return malaya.text.function.split_into_sentences(text)

    def word_tokenize(text: str) -> list[str]:
        return _word_tokenizer.tokenize(text)

    print("Using Malaya tokenizers.")

except ImportError:
    print("Malaya not installed -> using regex fallback tokenizers.")

    _ABBREV = {"dr", "sdn", "bhd", "no", "vs", "cth", "sk", "yb"}
    _SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9À-ɏ‘“])")

    def sentence_tokenize(text: str) -> list[str]:
        raw = _SENT_SPLIT_RE.split(text)
        sentences, buf = [], ""
        for part in raw:
            buf = f"{buf} {part}".strip() if buf else part
            last_word = re.findall(r"[A-Za-z]+", buf)[-1:] if buf else []
            if last_word and last_word[0].lower() in _ABBREV:
                continue  # don't split after a known abbreviation
            sentences.append(buf)
            buf = ""
        if buf:
            sentences.append(buf)
        return [s.strip() for s in sentences if s.strip()]

    _WORD_RE = re.compile(r"[\w']+|[.,!?;:\-–—()]", re.UNICODE)

    def word_tokenize(text: str) -> list[str]:
        return _WORD_RE.findall(text)


Using Malaya tokenizers.


In [13]:
work_df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE) if SAMPLE_SIZE else df.copy()
work_df = work_df.reset_index(drop=True)
print(f"Running tokenization+ on {len(work_df)} of {len(df)} rows.")

work_df["sentences"] = work_df["body_cased"].progress_apply(sentence_tokenize)
work_df["tokens"] = work_df["body_lower"].progress_apply(word_tokenize)
work_df["n_sentences"] = work_df["sentences"].str.len()
work_df["n_tokens"] = work_df["tokens"].str.len()

work_df[["title_cased", "n_sentences", "n_tokens"]].head(5)


Running tokenization+ on 8380 of 8380 rows.


100%|██████████| 8380/8380 [00:10<00:00, 767.90it/s]


,title_cased,n_sentences,n_tokens
0,'1 Inci Pun Boleh Lemas'- Ini Cara Elak Risiko...,19,328
1,'1 Kebaikan Dapat Upah RM1' - Lelaki Kongsi Te...,29,379
2,1 Minuman 2 Rasa - Susu Frog Bang Dari Tealive...,37,385
3,"Bukan Setakat Kena Baca Doa Sahaja, Ini 10 Ada...",33,1375
4,"10 Aktiviti DIY Di Rumah Untuk Anak, Lengkap S...",34,343


## 5b. Manual short-form / slang normalization

This corpus is written in casual, spoken-style Malay ("bahasa pasar"), full of short forms
and SMS-speak that a tokenizer/stemmer alone doesn't reliably resolve: `tak`→`tidak`, `kat`→`di`,
`je`→`sahaja`, `korang`→`kamu semua`, `diorang`→`mereka`, `dgn`→`dengan`, `utk`→`untuk`,
`dah`→`sudah`... Left unexpanded, lemmatization in section 7 just returns them as-is (the
stemmer works off the surface token — it can't invent an expansion), so `tak` and `tidak` end
up as two different tokens instead of one.

Malaya ships its own hand-curated normalizer for exactly this (`malaya.dictionary.rules_normalizer`,
~3,400 entries) — most common short forms are already covered by that, no manual work needed.
What's still genuinely **manual** is the long tail: short forms specific to this gaming-news
corpus, or ones ambiguous out of context (`kt` could be `kat` or something else), that a person
has to read in a real sentence to resolve correctly. The workflow:

1. Malaya's normalizer is loaded as the base dictionary and applied automatically below.
2. Tokens still short/frequent after that — filtered against real Malay *and* English
   dictionary words so brand names and legitimate short words don't clutter the list — are
   surfaced with an example sentence into `../data/processed/short_form_candidates.csv`.
3. You fill in `canonical_form` (and optionally `meaning`) by hand for genuine short forms,
   and leave it blank for anything that turns out to be a brand/product name.
4. A later cell re-loads your edits and re-applies the (now extended) dictionary before
   lemmatization runs.


In [14]:
# Base dictionary: Malaya ships its own hand-curated Malay short-form/slang normalizer
# (malaya.dictionary.rules_normalizer, ~3,400 entries) covering most of what you'd otherwise
# have to build by hand (tak, kat, je, korang, diorang, dgn, utk, mmg, org, jgn, sbb, yg, tp, ...).
# We start from that, patch a few gaps/no-ops it has, then layer your own manual additions on top.
try:
    import malaya

    SHORT_FORM_MAP = {
        k: v for k, v in malaya.dictionary.rules_normalizer.items()
        if v and v != k  # drop no-op entries (a few keys map to themselves or to None)
    }
    print(f"Loaded {len(SHORT_FORM_MAP)} short-form mappings from malaya.dictionary.rules_normalizer.")
except Exception:
    SHORT_FORM_MAP = {}
    print("Malaya not installed -> starting from an empty dictionary (seed a few manually below).")

# A handful of gaps/patches on top of Malaya's own normalizer.
SHORT_FORM_MAP.update({
    "sy": "saya", "sya": "saya", "aq": "saya", "ak": "saya",
    "bg": "bagi",
})


def apply_short_form_map(tokens: list[str]) -> list[str]:
    return [SHORT_FORM_MAP.get(t.lower(), t) for t in tokens]


work_df["tokens_raw"] = work_df["tokens"]  # keep the pre-expansion tokens for reference
work_df["tokens"] = work_df["tokens_raw"].apply(apply_short_form_map)

n_changed = sum(
    1 for raw, exp in zip(work_df["tokens_raw"], work_df["tokens"]) if raw != exp
)
print(f"Dictionary now has {len(SHORT_FORM_MAP)} entries.")
print(f"{n_changed} of {len(work_df)} docs had at least one token expanded.")


Loaded 3355 short-form mappings from malaya.dictionary.rules_normalizer.
Dictionary now has 3356 entries.
7807 of 8380 docs had at least one token expanded.


In [15]:
# Surface remaining short/informal-looking tokens that are still frequent, for manual review.
# Excludes: anything already in SHORT_FORM_MAP, standard stopwords, non-alphabetic tokens,
# AND real dictionary words in Malay or English (malaya.dictionary.MALAY_WORDS/ENGLISH_WORDS)
# -- otherwise this list fills up with legitimate short words and English/brand loanwords
# ("game", "pc", "xbox", "web", "the", "of") that aren't short forms needing expansion at all.
from collections import Counter

try:
    import malaya
    _known_stopwords = set(malaya.text.function.get_stopwords())
    _malay_words = malaya.dictionary.MALAY_WORDS
    _english_words = malaya.dictionary.ENGLISH_WORDS
except Exception:
    _known_stopwords = set()
    _malay_words = set()
    _english_words = set()

CANDIDATE_MAX_LEN = 4
CANDIDATE_MIN_COUNT = 5

token_counts = Counter(t.lower() for toks in work_df["tokens_raw"] for t in toks)

candidates = {
    tok: cnt for tok, cnt in token_counts.items()
    if tok.isalpha()
    and len(tok) <= CANDIDATE_MAX_LEN
    and cnt >= CANDIDATE_MIN_COUNT
    and tok not in SHORT_FORM_MAP
    and tok not in _known_stopwords
    and tok not in _malay_words
    and tok not in _english_words
}

# grab one example sentence per candidate, for context
example_sentence = {}
for sentences in work_df["sentences"]:
    for sent in sentences:
        sent_tokens = {w.lower().strip(".,!?;:") for w in sent.split()}
        for tok in candidates:
            if tok not in example_sentence and tok in sent_tokens:
                example_sentence[tok] = sent

candidates_df = pd.DataFrame(
    [
        {"token": tok, "frequency": cnt, "example_sentence": example_sentence.get(tok, ""),
         "canonical_form": "", "meaning": ""}
        for tok, cnt in sorted(candidates.items(), key=lambda kv: -kv[1])
    ]
)

candidates_path = OUT_DIR / "short_form_candidates.csv"
candidates_df.to_csv(candidates_path, index=False, encoding="utf-8-sig")
print(f"{len(candidates_df)} candidate short forms -> {candidates_path}")
print("Note: some entries will still be brand/product names (e.g. 'rog', 'pubg') rather than "
      "true slang -- leave canonical_form blank for those, they don't need expansion.")
print("Fill in 'canonical_form' (read 'example_sentence' for context), save, then run the merge cell below.")
candidates_df.head(15)


1390 candidate short forms -> ..\data\processed\short_form_candidates.csv
Note: some entries will still be brand/product names (e.g. 'rog', 'pubg') rather than true slang -- leave canonical_form blank for those, they don't need expansion.
Fill in 'canonical_form' (read 'example_sentence' for context), save, then run the merge cell below.


,token,frequency,example_sentence,canonical_form,meaning
0,م,2406,,,
1,ل,2166,,,
2,tips,2058,Umum mengetahui Hanis gemar berkongsi tips dan...,,
3,ن,2010,,,
4,ي,1986,,,
5,و,1816,J : Ayat 3 surah Al-maidah أليوم أكملت لكم دين...,,
6,a,1685,"Me: Eeeeeeii A few hours later, hati saya masi...",,
7,ه,1626,,,
8,ر,1501,,,
9,ا,1369,"Daripada Ummul Mukminin, Umm Salamah R.Anha ba...",,


Run this AFTER you've filled in 'canonical_form' in ../data/processed/short_form_candidates.csv and saved it.
Safe to re-run repeatedly as you extend the sheet -> just re-applies the growing dictionary.

In [16]:
# Run this AFTER you've filled in 'canonical_form' in ../data/processed/short_form_candidates.csv and saved it.
# Safe to re-run repeatedly as you extend the sheet -> just re-applies the growing dictionary.
if candidates_path.exists():
    _filled = pd.read_csv(candidates_path, encoding="utf-8-sig")
    _filled = _filled[_filled["canonical_form"].notna() & (_filled["canonical_form"].astype(str).str.strip() != "")]
    added = 0
    for _, r in _filled.iterrows():
        tok = str(r["token"]).lower()
        if tok not in SHORT_FORM_MAP:
            added += 1
        SHORT_FORM_MAP[tok] = str(r["canonical_form"]).strip()
    print(f"Added/updated {len(_filled)} entries from {candidates_path} ({added} new). Dictionary now has {len(SHORT_FORM_MAP)} entries.")

work_df["tokens"] = work_df["tokens_raw"].apply(apply_short_form_map)
n_changed = sum(1 for raw, exp in zip(work_df["tokens_raw"], work_df["tokens"]) if raw != exp)
print(f"{n_changed} of {len(work_df)} docs now have at least one token expanded.")
print("Re-run section 6 (stopwords) and 7 (lemmatization) below so they pick up the expanded tokens.")


Added/updated 0 entries from ..\data\processed\short_form_candidates.csv (0 new). Dictionary now has 3356 entries.
7807 of 8380 docs now have at least one token expanded.
Re-run section 6 (stopwords) and 7 (lemmatization) below so they pick up the expanded tokens.


## 6. Stopword removal (optional)

The cleaned text here is meant to feed **embeddings / language models**, not a classic
BoW/TF-IDF pipeline — stopwords carry real distributional signal for those models, so
they are **not removed from the main output**. This step is built anyway as an
opt-in side artifact (`tokens_no_stop`), controlled by `DO_STOPWORD_REMOVAL`, in case a
downstream classic-IR use case ever needs it.


In [17]:
# try:
#     import malaya
#     STOPWORDS = set(malaya.text.function.get_stopwords())
#     print(f"Loaded {len(STOPWORDS)} stopwords from Malaya.")
# except Exception:
#     STOPWORDS = {
#         "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "ini", "itu",
#         "dalam", "tidak", "akan", "pada", "juga", "adalah", "atau", "kami",
#         "anda", "saya", "kita", "ada", "tak", "nak", "kat", "je", "lah",
#         "pun", "tu", "ni", "dia", "kalau", "sudah", "boleh", "sebagai",
#     }
#     print(f"Malaya unavailable -> using {len(STOPWORDS)}-word builtin fallback list.")


# def remove_stopwords(tokens: list[str]) -> list[str]:
#     return [t for t in tokens if t.lower() not in STOPWORDS]


# if DO_STOPWORD_REMOVAL:
#     work_df["tokens_no_stop"] = work_df["tokens"].progress_apply(remove_stopwords)
# else:
#     print("DO_STOPWORD_REMOVAL=False -> skipping (tokens_no_stop not created). "
#           "Main pipeline output keeps stopwords intact for embedding/LM use.")


## 7. Lemmatization (Malaya)

Uses `malaya.stem` — the fast rule/dictionary-based Sastrawi-derived stemmer
(`malaya.stem.sastrawi()`) by default, which also performs lemmatization for Malay/Indonesian.
A HuggingFace seq2seq model (`malaya.stem.huggingface()`) is available for higher accuracy at
much higher compute cost — swap it in below if quality matters more than speed.

Repeated tokens (very common in a gaming-news corpus — "game", "pemain", "PS5", ...) are
cached with `lru_cache` so the model is only invoked once per unique lowercase token.


In [18]:
try:
    import malaya

    # _stemmer = malaya.stem.sastrawi()
    # For higher quality (slower, needs a GPU/decent CPU + more RAM):
    _stemmer = malaya.stem.huggingface()

    @functools.lru_cache(maxsize=200_000)
    def lemmatize_token(token: str) -> str:
        if not re.search(r"[a-zA-Z]", token):
            return token  # punctuation/numbers pass through unchanged
        try:
            return _stemmer.stem(token)
        except Exception:
            return token

    print("Using Malaya sastrawi stemmer/lemmatizer.")

except ImportError:
    print("Malaya not installed -> lemmatization step will pass tokens through unchanged.")
    print("Install with: pip install malaya malaya-boilerplate torch")

    @functools.lru_cache(maxsize=200_000)
    def lemmatize_token(token: str) -> str:
        return token


def lemmatize_tokens(tokens: list[str]) -> list[str]:
    return [lemmatize_token(t) for t in tokens]


work_df["lemmas"] = work_df["tokens"].progress_apply(lemmatize_tokens)
work_df[["tokens", "lemmas"]].head(3)


Using Malaya sastrawi stemmer/lemmatizer.


100%|██████████| 8380/8380 [14:31<00:00,  9.62it/s] 


,tokens,lemmas
0,"[insiden, kanak-kanak, lemas, di, dalam, bilik...","[insiden, kanak-kanak, lemas, di, dalam, bilik..."
1,"["", papa, ,, papa.jom, pergi, kedai, nak, beli...","["", papa, ,, papajomk, pergi, kedai, nak, bel,..."
2,"[ha, !, kalau, nak, tahu, baru-baru, ini, admi...","[ha, !, kalau, nak, tahu, baru-baru, ini, admi..."


## 8. Assemble & save the cleaned corpus

Final columns per document: identifiers, both cased/lowercased normalized text (for
embeddings vs. NER), sentence/token/lemma arrays, and both tag layers.


In [19]:
final_cols = [
    "url", "title_cased", "title_lower",
    "body_cased", "body_lower",
    "sentences", "tokens_raw", "tokens", "lemmas",
    "n_sentences", "n_tokens",
    "lang",
]
final_df = work_df[final_cols].copy()

out_jsonl = OUT_DIR / "siraplimau_cleaned.jsonl"
with out_jsonl.open("w", encoding="utf-8") as f:
    for _, row in final_df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

out_parquet = OUT_DIR / "siraplimau_cleaned.parquet"
final_df.to_parquet(out_parquet, index=False)

print(f"Saved {len(final_df)} cleaned docs:")
print(f"  {out_jsonl}")
print(f"  {out_parquet}")
print(f"  {candidates_path}  (short-form dictionary candidates)")
final_df.head(3)


Saved 8380 cleaned docs:
  ..\data\processed\siraplimau_cleaned.jsonl
  ..\data\processed\siraplimau_cleaned.parquet
  ..\data\processed\short_form_candidates.csv  (short-form dictionary candidates)


,url,title_cased,title_lower,body_cased,body_lower,sentences,tokens_raw,tokens,lemmas,n_sentences,n_tokens,lang
0,https://siraplimau.com/1-inci-pun-boleh-lemas-...,'1 Inci Pun Boleh Lemas'- Ini Cara Elak Risiko...,'1 inci pun boleh lemas'- ini cara elak risiko...,Insiden kanak-kanak lemas di dalam bilik air b...,insiden kanak-kanak lemas di dalam bilik air b...,[Insiden kanak-kanak lemas di dalam bilik air ...,"[insiden, kanak-kanak, lemas, di, dalam, bilik...","[insiden, kanak-kanak, lemas, di, dalam, bilik...","[insiden, kanak-kanak, lemas, di, dalam, bilik...",19,328,ms
1,https://siraplimau.com/1-kebaikan-dapat-upah-r...,'1 Kebaikan Dapat Upah RM1' - Lelaki Kongsi Te...,'1 kebaikan dapat upah rm1' - lelaki kongsi te...,"""Papa,papa.jom pergi kedai nak beli mainan.jom...","""papa,papa.jom pergi kedai nak beli mainan.jom...","[""Papa,papa., jom pergi kedai nak beli mainan....","["", papa, ,, papa.jom, pergi, kedai, nak, beli...","["", papa, ,, papa.jom, pergi, kedai, nak, beli...","["", papa, ,, papajomk, pergi, kedai, nak, bel,...",29,379,ms
2,https://siraplimau.com/1-minuman-2-rasa-susu-f...,1 Minuman 2 Rasa - Susu Frog Bang Dari Tealive...,1 minuman 2 rasa - susu frog bang dari tealive...,Ha! Kalau nak tahu baru-baru ni admin dijemput...,ha! kalau nak tahu baru-baru ni admin dijemput...,[Kalau nak tahu baru-baru ni admin dijemput ol...,"[ha, !, kalau, nak, tahu, baru-baru, ni, admin...","[ha, !, kalau, nak, tahu, baru-baru, ini, admi...","[ha, !, kalau, nak, tahu, baru-baru, ini, admi...",37,385,ms


## 9. Interactive word → lemma lookup

Everything above processes the whole corpus in batch. This section is the actual end-user
tool: type a word (or a short phrase), get its lemma back — reusing the exact same
normalization → short-form expansion → lemmatization steps the pipeline used, so results
are consistent with what's in `gamersantai_cleaned.jsonl`.

Three pieces:
1. `lemmatize_word(text)` — the reusable function, single word or short phrase in, list of
   `(surface_form, lemma)` pairs out. Handles unseen/out-of-vocabulary input gracefully (the
   short-form dictionary and stemmer both just pass unrecognized tokens through unchanged
   rather than erroring).
2. A saved `word_lemma_dictionary.json` — every unique `token → lemma` mapping seen while
   processing the corpus, for instant dictionary-style lookup without needing Malaya/PyTorch
   loaded at all (useful if you want this lookup in a lighter-weight place later, e.g. a
   small script or web app).
3. An interactive prompt cell to actually type words into and get lemmas back, right here in
   the notebook.


In [20]:
def lemmatize_word(text: str) -> list[tuple[str, str]]:
    """Run arbitrary user input through the same normalize -> tokenize -> short-form
    expand -> lemmatize pipeline used on the corpus. Returns [(surface_form, lemma), ...].
    Works on a single word or a short phrase; unrecognized tokens pass through unchanged
    rather than erroring (typos, brand names, code-switched English, etc.).
    """
    text = normalize_text(text).lower()
    tokens = word_tokenize(text)
    tokens = apply_short_form_map(tokens)
    return [(tok, lemmatize_token(tok)) for tok in tokens]


# Quick self-check against a few known short forms + a plain word + something unseen.
for example in ["bermain", "tak", "kat", "korang", "sebuahrumah123"]:
    print(f"{example!r:20} -> {lemmatize_word(example)}")


'bermain'            -> [('bermain', 'main')]
'tak'                -> [('tidak', 'tidak')]
'kat'                -> [('di', 'di')]
'korang'             -> [('kamu semua', 'kamu semua')]
'sebuahrumah123'     -> [('sebuahrumah123', 'sebuahrumah')]


In [21]:
# Aggregate every unique token -> lemma pair seen across the processed corpus into a plain
# lookup table. This is a fast fallback path for words already seen in the corpus (dict lookup,
# no model call) and a portable artifact if you want this lookup somewhere lighter-weight than
# a notebook with Malaya/PyTorch loaded (a small script, a web app, etc.).
word_lemma_dictionary = {}
for toks, lems in zip(final_df["tokens_raw"], final_df["lemmas"]):
    for tok, lem in zip(toks, lems):
        word_lemma_dictionary.setdefault(tok, lem)

dict_path = OUT_DIR / "word_lemma_dictionary.json"
dict_path.write_text(json.dumps(word_lemma_dictionary, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"{len(word_lemma_dictionary)} unique word -> lemma pairs -> {dict_path}")


95757 unique word -> lemma pairs -> ..\data\processed\word_lemma_dictionary.json


In [1]:
# Interactive lookup widget. Type a word (or short phrase) and press Enter -> lemma(s) appear below.
# Falls back to a plain input()-loop if ipywidgets isn't available (input() can be unreliable
# in some notebook frontends -- if you don't get a prompt either way, just call
# lemmatize_word("your word") directly in a new cell instead).
try:
    import ipywidgets as widgets
    from IPython.display import display

    word_in = widgets.Text(
        description="Word:", placeholder="type a word, press Enter", continuous_update=False
    )
    result_out = widgets.Output()

    def _on_change(change):
        if change["name"] != "value":
            return
        with result_out:
            result_out.clear_output()
            text = change["new"].strip()
            if not text:
                return
            for surface, lemma in lemmatize_word(text):
                marker = " (unchanged)" if surface == lemma else ""
                print(f"  {surface!r:20} -> {lemma!r}{marker}")

    word_in.observe(_on_change, names="value")
    display(widgets.VBox([word_in, result_out]))

except ImportError:
    print("ipywidgets not available -> falling back to a plain input() loop.")
    print("If no prompt appears below either, just call lemmatize_word('your word') directly instead.")
    while True:
        try:
            user_input = input("Enter a word (or 'quit'): ").strip()
        except Exception:
            break
        if not user_input or user_input.lower() == "quit":
            break
        for surface, lemma in lemmatize_word(user_input):
            marker = " (unchanged)" if surface == lemma else ""
            print(f"  {surface!r:20} -> {lemma!r}{marker}")


ipywidgets not available -> falling back to a plain input() loop.
If no prompt appears below either, just call lemmatize_word('your word') directly instead.


NameError: name 'lemmatize_word' is not defined

**If the widget above doesn't render** (this can happen right after a kernel restart — the
`ipywidgets` comm channel sometimes doesn't reconnect on the very first render; re-running the
cell a second time, or waiting for the kernel to fully finish restarting first, usually fixes
it): skip the UI entirely and just edit `WORD_TO_LOOKUP` below and re-run — no widget, no
`input()`, nothing that depends on the notebook frontend. This always works.

In [ ]:
WORD_TO_LOOKUP = "kau pa hal ni"  # <- edit this, then re-run this cell

for surface, lemma in lemmatize_word(WORD_TO_LOOKUP):
    marker = " (unchanged)" if surface == lemma else ""
    print(f"  {surface!r:20} -> {lemma!r}{marker}")


  'kamu'               -> 'kamu' (unchanged)
  'pa'                 -> 'pa' (unchanged)
  'hal'                -> 'hal' (unchanged)
  'ini'                -> 'ini' (unchanged)
